# Preprocessing: Annotate Treatment Patterns

Demonstrates `annotate_treatment_patterns`, which classifies each patient into one or more of 20 boolean treatment pattern flags based on their treatment timeline relative to the diagnosis date.

| # | Column suffix | Pattern |
|---|---|---|
| 1 | `only_surgery` | Surgery only |
| 2 | `only_radio` | Radiotherapy only |
| 3 | `only_chemo` | Chemotherapy only |
| 4 | `only_immuno` | Immunotherapy only |
| 5 | `only_target` | Targeted therapy only |
| 6 | `concomitant_systemic_radio` | Concomitant systemic + radiotherapy |
| 7 | `surgery_postop_radio` | Surgery + post-operative radiotherapy |
| 8 | `surgery_adj_chemo` | Surgery + adjuvant chemotherapy |
| 9 | `surgery_postop_radio_concomi_chemo` | Surgery + post-op radio + concomitant chemo |
| 10 | `radio_adj_chemo` | Radiotherapy + adjuvant chemotherapy |
| 11 | `concomi_chemo_radio_adj_chemo` | Concomitant chemo-radio + adjuvant chemo |
| 12 | `chemo_immuno` | Chemo + immunotherapy |
| 13 | `chemo_target` | Chemo + targeted therapy |
| 14 | `immuno_target` | Immuno + targeted therapy |
| 15 | `neoadj_chemo_radio` | Neoadjuvant chemo → radiotherapy |
| 16 | `neoadj_chemo_surgery` | Neoadjuvant chemo → surgery |
| 17 | `neoadj_chemo_concomi_chemo_radio` | Neoadj chemo → concomitant chemo-radio |
| 18 | `neoadj_chemo_concomi_chemo_radio_adj_chemo` | Neoadj chemo → concomitant chemo-radio → adj chemo |
| 19 | `neoadj_chemo_radio_adj_chemo` | Neoadj chemo → radio → adjuvant chemo |
| 20 | `other` | None of the above |


In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

pd.options.display.max_columns = None
pd.options.display.width = 200

## Create test data

One synthetic patient per rule (20 total), split across two mock nodes.  
Dates are chosen so that each patient matches exactly the labelled rule and no other.
The same date values are used as in the unit test suite (`test_treatment_patterns.py`).

In [ ]:
NAT = pd.NaT
TZ  = "UTC"

def ts(s):
    return pd.Timestamp(s, tz=TZ)

def base(patient_id, expected_pattern):
    """Empty row — diagnosis only, all treatment slots NaT."""
    return {
        "patient_id":             patient_id,
        "expected_pattern":       expected_pattern,
        "diagnosis_date":         ts("2020-01-01"),
        # surgery
        "surgery_1_date":         NAT,
        # radiotherapy
        "radio_1_start_date":     NAT, "radio_1_end_date":    NAT,
        # chemotherapy — up to 3 lines (rule 18 needs 3)
        "chemo_1_start_date":     NAT, "chemo_1_end_date":    NAT,
        "chemo_2_start_date":     NAT, "chemo_2_end_date":    NAT,
        "chemo_3_start_date":     NAT, "chemo_3_end_date":    NAT,
        # immunotherapy
        "immuno_1_start_date":    NAT, "immuno_1_end_date":   NAT,
        # targeted therapy
        "targeted_1_start_date":  NAT, "targeted_1_end_date": NAT,
    }

rows = [
    # ── Single-modality rules ───────────────────────────────────────────────

    # 1  only_surgery
    {**base(1, "only_surgery"),
     "surgery_1_date": ts("2020-02-15")},                          # +45 d

    # 2  only_radio
    {**base(2, "only_radio"),
     "radio_1_start_date": ts("2020-02-15"), "radio_1_end_date": ts("2020-03-15")},

    # 3  only_chemo  (single line)
    {**base(3, "only_chemo"),
     "chemo_1_start_date": ts("2020-02-01"), "chemo_1_end_date": ts("2020-04-01")},

    # 4  only_immuno  (single line)
    {**base(4, "only_immuno"),
     "immuno_1_start_date": ts("2020-02-01"), "immuno_1_end_date": ts("2020-04-01")},

    # 5  only_target  (single line)
    {**base(5, "only_target"),
     "targeted_1_start_date": ts("2020-02-01"), "targeted_1_end_date": ts("2020-04-01")},

    # ── Combined rules ──────────────────────────────────────────────────────

    # 6  concomitant_systemic_radio
    #    Radio starts first; chemo starts 5 d later (within the 14-d gap threshold).
    #    chemo_start > radio_start prevents neoadj_chemo_radio from firing.
    {**base(6, "concomitant_systemic_radio"),
     "radio_1_start_date": ts("2020-02-05"), "radio_1_end_date": ts("2020-03-20"),
     "chemo_1_start_date": ts("2020-02-10"), "chemo_1_end_date": ts("2020-03-25")},

    # 7  surgery_postop_radio
    #    Surgery day+31; radio starts 59 d after surgery (< 120 d threshold).
    {**base(7, "surgery_postop_radio"),
     "surgery_1_date":     ts("2020-02-01"),
     "radio_1_start_date": ts("2020-04-01"), "radio_1_end_date": ts("2020-05-01")},

    # 8  surgery_adj_chemo
    #    Surgery day+31; chemo starts 89 d after surgery (< 120 d threshold).
    {**base(8, "surgery_adj_chemo"),
     "surgery_1_date":     ts("2020-02-01"),
     "chemo_1_start_date": ts("2020-05-01"), "chemo_1_end_date": ts("2020-07-01")},

    # 9  surgery_postop_radio_concomi_chemo
    #    Surgery day+31; radio + chemo start together 59 d later (|start gap|=4 d, |end gap|=4 d).
    {**base(9, "surgery_postop_radio_concomi_chemo"),
     "surgery_1_date":     ts("2020-02-01"),
     "radio_1_start_date": ts("2020-04-01"), "radio_1_end_date": ts("2020-05-01"),
     "chemo_1_start_date": ts("2020-04-05"), "chemo_1_end_date": ts("2020-04-27")},

    # 10 radio_adj_chemo
    #    Radio day+31–+91; chemo starts 61 d after radio end (< 120 d threshold).
    {**base(10, "radio_adj_chemo"),
     "radio_1_start_date": ts("2020-02-01"), "radio_1_end_date": ts("2020-04-01"),
     "chemo_1_start_date": ts("2020-06-01"), "chemo_1_end_date": ts("2020-08-01")},

    # 11 concomi_chemo_radio_adj_chemo
    #    chemo1 starts on the same day as radio (prevents neoadj variants);
    #    chemo2 starts 28 d after max(chemo1_end, radio_end) = Apr 3.
    {**base(11, "concomi_chemo_radio_adj_chemo"),
     "radio_1_start_date":  ts("2020-02-05"), "radio_1_end_date":  ts("2020-04-03"),
     "chemo_1_start_date":  ts("2020-02-05"), "chemo_1_end_date":  ts("2020-04-01"),
     "chemo_2_start_date":  ts("2020-05-01"), "chemo_2_end_date":  ts("2020-07-01")},

    # ── Multi-systemic rules ────────────────────────────────────────────────

    # 12 chemo_immuno
    {**base(12, "chemo_immuno"),
     "chemo_1_start_date":  ts("2020-02-01"), "chemo_1_end_date":  ts("2020-04-01"),
     "immuno_1_start_date": ts("2020-02-15"), "immuno_1_end_date": ts("2020-05-01")},

    # 13 chemo_target
    {**base(13, "chemo_target"),
     "chemo_1_start_date":     ts("2020-02-01"), "chemo_1_end_date":     ts("2020-04-01"),
     "targeted_1_start_date":  ts("2020-02-15"), "targeted_1_end_date":  ts("2020-05-01")},

    # 14 immuno_target
    {**base(14, "immuno_target"),
     "immuno_1_start_date":   ts("2020-02-01"), "immuno_1_end_date":   ts("2020-04-01"),
     "targeted_1_start_date": ts("2020-02-15"), "targeted_1_end_date": ts("2020-05-01")},

    # ── Neoadjuvant rules ───────────────────────────────────────────────────

    # 15 neoadj_chemo_radio
    #    Chemo ends Apr 1; radio starts May 1 (30 d later, within 90-d threshold).
    {**base(15, "neoadj_chemo_radio"),
     "chemo_1_start_date": ts("2020-02-01"), "chemo_1_end_date": ts("2020-04-01"),
     "radio_1_start_date": ts("2020-05-01"), "radio_1_end_date": ts("2020-06-01")},

    # 16 neoadj_chemo_surgery
    #    Chemo ends Apr 1; surgery May 1 (30 d later, within 90-d threshold).
    {**base(16, "neoadj_chemo_surgery"),
     "chemo_1_start_date": ts("2020-02-01"), "chemo_1_end_date": ts("2020-04-01"),
     "surgery_1_date":     ts("2020-05-01")},

    # 17 neoadj_chemo_concomi_chemo_radio
    #    chemo1 (neoadj), then chemo2 + radio together
    #    (chemo2 starts 4 d after radio_start; |end gap| = 4 d).
    {**base(17, "neoadj_chemo_concomi_chemo_radio"),
     "chemo_1_start_date":  ts("2020-02-01"), "chemo_1_end_date":  ts("2020-04-01"),
     "radio_1_start_date":  ts("2020-05-01"), "radio_1_end_date":  ts("2020-07-01"),
     "chemo_2_start_date":  ts("2020-05-05"), "chemo_2_end_date":  ts("2020-06-27")},

    # 18 neoadj_chemo_concomi_chemo_radio_adj_chemo
    #    Same as rule 17 + chemo3 starting 31 d after max(chemo2_end=Jun27, radio_end=Jul1) = Jul 1.
    {**base(18, "neoadj_chemo_concomi_chemo_radio_adj_chemo"),
     "chemo_1_start_date":  ts("2020-02-01"), "chemo_1_end_date":  ts("2020-04-01"),
     "radio_1_start_date":  ts("2020-05-01"), "radio_1_end_date":  ts("2020-07-01"),
     "chemo_2_start_date":  ts("2020-05-05"), "chemo_2_end_date":  ts("2020-06-27"),
     "chemo_3_start_date":  ts("2020-08-01"), "chemo_3_end_date":  ts("2020-10-01")},

    # 19 neoadj_chemo_radio_adj_chemo
    #    chemo1 (neoadj) → radio → chemo2 (adjuvant, starts after radio ends, no overlap).
    {**base(19, "neoadj_chemo_radio_adj_chemo"),
     "chemo_1_start_date":  ts("2020-02-01"), "chemo_1_end_date":  ts("2020-04-01"),
     "radio_1_start_date":  ts("2020-05-01"), "radio_1_end_date":  ts("2020-06-15"),
     "chemo_2_start_date":  ts("2020-07-15"), "chemo_2_end_date":  ts("2020-09-15")},

    # ── Catch-all ───────────────────────────────────────────────────────────

    # 20 other — surgery + chemo both > 90 d after diagnosis
    {**base(20, "other"),
     "surgery_1_date":     ts("2020-08-01"),   # +212 d
     "chemo_1_start_date": ts("2020-09-01"), "chemo_1_end_date": ts("2020-11-01")},
]

test_data = pd.DataFrame(rows).reset_index(drop=True)
print(f"Test DataFrame: {len(test_data)} patients (one per rule)")
test_data[["patient_id", "expected_pattern"]]

## Set up MockNetwork

Patients are distributed across two nodes to simulate a federated environment.

In [ ]:
from vantage6.algorithm.mock.network import MockNetwork

network = MockNetwork(
    "v6_preprocessing",
    datasets=[
        {"cohort_1": {"database": test_data.iloc[:10].copy(), "db_type": "omop"}},
        {"cohort_1": {"database": test_data.iloc[10:].copy(), "db_type": "omop"}},
    ],
    collaboration_id=1,
)

client = network.user_client

## Run annotate_treatment_patterns

All parameters are shown explicitly; any parameter can be omitted to use its default.

In [ ]:
client.dataframe.preprocess(
    id_=1,
    method="annotate_treatment_patterns",
    image="v6-preprocessing",
    arguments={
        "prefix":                          "trt_pattern_",
        "general_rule_days":                90,
        "concomitant_start_gap":            14,
        "concomitant_end_gap":              14,
        "surgery_postop_radio_days":        120,
        "surgery_adjuvant_chemo_days":      120,
        "postop_radio_concomi_start_gap":   14,
        "postop_radio_concomi_end_gap":     14,
        "radio_adjuvant_chemo_days":        120,
        "concomi_radio_adj_start_gap":      14,
        "concomi_radio_adj_end_gap":        14,
        "concomi_radio_adj_to_next":        90,
        "chemo_immuno_days":                180,
        "neoadj_chemo_to_radio":            90,
        "neoadj_chemo_to_surgery":          90,
        "neoadj_concomi_to_phase":          90,
        "neoadj_concomi_chemo2_start_gap":  14,
        "neoadj_concomi_chemo2_end_gap":    14,
        "neoadj_concomi_adj_to_next":       90,
        "neoadj_radio_adj_chemo1_to_radio": 90,
        "neoadj_radio_adj_chemo2_to_chemo": 90,
    }
)

## Results

Collect both nodes and show which pattern each patient matched.

In [ ]:
df_node1 = client.network.get_node(1).dataframes["cohort_1"]
df_node2 = client.network.get_node(2).dataframes["cohort_1"]

pattern_cols = [c for c in df_node1.columns if c.startswith("trt_pattern_")]
print(f"{len(pattern_cols)} pattern columns added")

# Combine both nodes for a single overview
df_all = pd.concat([df_node1, df_node2], ignore_index=True)

df_all["matched_patterns"] = df_all[pattern_cols].apply(
    lambda row: [c.replace("trt_pattern_", "") for c in pattern_cols if row[c]], axis=1
)
df_all["correct"] = df_all.apply(
    lambda row: row["expected_pattern"] in row["matched_patterns"], axis=1
)

df_all[["patient_id", "expected_pattern", "matched_patterns", "correct"]]

## Validation summary

In [ ]:
n_correct = df_all["correct"].sum()
n_total   = len(df_all)
print(f"{n_correct}/{n_total} patients matched their expected pattern")

mismatches = df_all[~df_all["correct"]][["patient_id", "expected_pattern", "matched_patterns"]]
if mismatches.empty:
    print("No mismatches — all rules fire as expected.")
else:
    print("\nMismatches:")
    display(mismatches)

## Full boolean flag matrix

Shows the raw boolean values for all 20 pattern columns across all patients.

In [ ]:
df_all[["patient_id", "expected_pattern"] + pattern_cols].set_index("patient_id")

## Effect of changing day thresholds

Patient 8 (`surgery_adj_chemo`) has chemo starting **89 days** after surgery.  
With the default `surgery_adjuvant_chemo_days=120` it matches `surgery_adj_chemo`.  
Tightening the threshold to **80 days** pushes it into `other`.

In [ ]:
network2 = MockNetwork(
    "v6_preprocessing",
    datasets=[
        {"cohort_1": {"database": test_data[test_data.patient_id == 8].copy(), "db_type": "omop"}},
    ],
    collaboration_id=1,
)
client2 = network2.user_client

client2.dataframe.preprocess(
    id_=1,
    method="annotate_treatment_patterns",
    image="v6-preprocessing",
    arguments={"surgery_adjuvant_chemo_days": 80},
)

df_tight = client2.network.get_node(1).dataframes["cohort_1"]
tight_cols = [c for c in df_tight.columns if c.startswith("trt_pattern_")]
matched = [c.replace("trt_pattern_", "") for c in tight_cols if df_tight[c].iloc[0]]
print(f"Patient 8 with surgery_adjuvant_chemo_days=120 (default) → surgery_adj_chemo")
print(f"Patient 8 with surgery_adjuvant_chemo_days=80  (tight)   → {matched}")